# 02 — Qwen3.5-35B-A3B Inference

Load **[Qwen/Qwen3.5-35B-A3B](https://huggingface.co/Qwen/Qwen3.5-35B-A3B)** and
run text generation on LUMI.

### Architecture recap
- **35B** total params, **~3B active** per forward pass (MoE: 8 routed + 1 shared out of 256 experts)
- Gated DeltaNet layers (linear attention) interleaved with sparse-MoE
- **Thinking mode** enabled by default — model outputs `<think>...</think>` before the answer
- Native context: 262 K tokens

**Requirements:** ≥ 1× MI250X (64 GB) or equivalent. BF16 precision recommended.

## 0. Imports & Configuration

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_ID   = 'Qwen/Qwen3.5-35B-A3B'
DTYPE      = torch.bfloat16          # BF16: best on AMD MI250X / NVIDIA Ampere+
DEVICE_MAP = 'auto'                  # spread across all visible GPUs
MAX_NEW_TOKENS = 512

# Optional: point to a local cache to avoid re-downloading
# os.environ['HF_HOME'] = '/scratch/<project>/hf_cache'
# ───────────────────────────────────────────────────────────────────────────

print(f'Device       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Model        : {MODEL_ID}')
print(f'Dtype        : {DTYPE}')

## 1. Load Tokenizer & Model

In [ ]:
print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
print('Tokenizer loaded.')

print('Loading model (this may take several minutes on first run) ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
    trust_remote_code=True,
)
model.eval()
print('Model loaded.')

# Print parameter count
total_params  = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters : {total_params / 1e9:.2f}B')

## 2. Helper: Generate Response

In [ ]:
def generate(
    prompt: str,
    system: str = 'You are a helpful assistant.',
    thinking_mode: bool = True,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = 0.7,
    top_p: float = 0.95,
    top_k: int = 20,
) -> str:
    """Run inference and return the model's answer (with optional think trace)."""
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': prompt},
    ]
    # enable_thinking controls <think>...</think> output
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking_mode,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    new_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

## 3. Basic Inference Examples

In [ ]:
# ── Example 1: simple question ─────────────────────────────────────────────
response = generate(
    prompt='What is the capital of Finland, and what is it famous for?',
    thinking_mode=False,   # fast, no reasoning trace
)
print('=== Response ===')
print(response)

In [ ]:
# ── Example 2: reasoning / thinking mode ───────────────────────────────────
math_response = generate(
    prompt='If a train travels 120 km in 1.5 hours, then slows down and '
           'covers the next 80 km in 2 hours, what is the average speed '
           'for the entire journey?',
    thinking_mode=True,    # model shows its reasoning inside <think>...</think>
    temperature=1.0,
    top_k=20,
)
print('=== Reasoning Response ===')
print(math_response)

In [ ]:
# ── Example 3: code generation ─────────────────────────────────────────────
code_response = generate(
    prompt='Write a Python function that computes the nth Fibonacci number '
           'using memoization. Include type hints and a docstring.',
    system='You are an expert Python programmer. Output clean, idiomatic code.',
    thinking_mode=False,
    temperature=0.6,
)
print('=== Code Response ===')
print(code_response)

## 4. GPU Memory Report

In [ ]:
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1024**3
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f'GPU {i}: {alloc:.2f} / {total:.2f} GB allocated')

---
Inference confirmed working. Proceed to **03_finetune_lora.ipynb** to fine-tune the model.